# 27 · Capstone: churn model from Delta to MLflow predictions

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** databricks. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Combine data quality, training, validation, tracking, batch scoring and an output contract.


## Step 1 — Generate and persist training data

This is a bounded demo of 1,000 rows; only this explicitly capped dataset is collected to the driver. Larger jobs should use distributed training and inference.


In [ ]:
import hashlib
from pyspark.sql import functions as F
user = spark.sql('SELECT current_user()').first()[0]
catalog = spark.sql('SELECT current_catalog()').first()[0]
schema = 'ai_lab_' + hashlib.sha256(user.encode()).hexdigest()[:10]
# Uses the current catalog; change it to a catalog where you can CREATE SCHEMA.
namespace = f'`{catalog}`.`{schema}`'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {namespace}')
def table(name):
    assert name.replace('_', '').isalnum()
    return f'{namespace}.`{name}`'
print('Course namespace:', namespace)

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                           weights=[0.75, 0.25], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
user=spark.sql('SELECT current_user()').first()[0]
mlflow.set_tracking_uri('databricks')
mlflow.set_experiment(f'/Users/{user}/ai_engineering_hands_on')

rows=[tuple([int(i)]+[float(v) for v in row]+[int(label)]) for i,(row,label) in enumerate(zip(X,y))]
ddl='id long, '+', '.join(f'f{i} double' for i in range(10))+', label int'
source=spark.createDataFrame(rows,ddl)
source.write.format('delta').mode('overwrite').saveAsTable(table('capstone_features'))
dataset=spark.table(table('capstone_features'))
assert dataset.count()==1000
display(dataset.limit(5))


**Check your result:** A course-owned Delta feature table contains 1,000 labeled rows.


## Step 2 — Read a bounded snapshot and split

Sorting by stable id makes the local split repeatable. In real churn prediction, use an as-of feature timestamp and a future outcome window to prevent leakage.


In [ ]:
snapshot=spark.sql(f"DESCRIBE HISTORY {table('capstone_features')}").select('version').first()[0]
bounded=spark.sql(f"SELECT * FROM {table('capstone_features')} VERSION AS OF {snapshot} ORDER BY id LIMIT 1001").collect()
assert len(bounded)<=1000, 'Demo collection cap exceeded'
features=np.array([[r[f'f{i}'] for i in range(10)] for r in bounded])
labels=np.array([r['label'] for r in bounded])
fit_X,test_X,fit_y,test_y=train_test_split(features,labels,stratify=labels,random_state=42)
train_X,val_X,train_y,val_y=train_test_split(fit_X,fit_y,stratify=fit_y,random_state=7)
print('train/validation/test:',len(train_y),len(val_y),len(test_y))


**Check your result:** Training, validation and test rows are distinct.


## Step 3 — Select and log a candidate

Use validation AUC for selection. Log the Delta version and the fitted model signature so results have traceable inputs.


In [ ]:
candidates=[]
for C in [0.1,1.,10.]:
    candidate=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=500)).fit(train_X,train_y)
    candidates.append((roc_auc_score(val_y,candidate.predict_proba(val_X)[:,1]),C))
best_auc,best_C=max(candidates)
final=make_pipeline(StandardScaler(),LogisticRegression(C=best_C,max_iter=500)).fit(fit_X,fit_y)
test_auc=roc_auc_score(test_y,final.predict_proba(test_X)[:,1])
assert best_auc>=0.7, 'Validation quality gate failed'
with mlflow.start_run(run_name='capstone_churn') as run:
    mlflow.log_params({'C':best_C,'seed':42,'delta_version':snapshot})
    mlflow.set_tag('source_table',f'{catalog}.{schema}.capstone_features')
    mlflow.log_metrics({'validation_auc':best_auc,'test_auc':test_auc})
    info=mlflow.sklearn.log_model(final,artifact_path='model',signature=infer_signature(fit_X,final.predict(fit_X)),input_example=fit_X[:2])
    run_id=run.info.run_id
print('Run:',run_id,'test AUC:',test_auc)


**Check your result:** One final model is logged after validation selection.


## Step 4 — Score and persist a bounded batch

This capstone uses local scoring for the 30-row demo; lab 14 provides a distributed UDF alternative. Reload before scoring to check packaging.


In [ ]:
loaded=mlflow.sklearn.load_model(info.model_uri)
batch_X=test_X[:30]
probability=loaded.predict_proba(batch_X)[:,1]
out=spark.createDataFrame([(int(i),float(p),int(p>=0.5),run_id) for i,p in enumerate(probability)],'batch_row long, probability double, prediction int, run_id string')
out=out.withColumn('scored_at',F.current_timestamp())
out.write.format('delta').mode('overwrite').saveAsTable(table('capstone_predictions'))
saved=spark.table(table('capstone_predictions'))
assert saved.count()==30
assert saved.filter((F.col('probability')<0)|(F.col('probability')>1)|F.col('probability').isNull()).count()==0
display(saved.limit(5))


**Check your result:** Thirty rows satisfy the output probability contract. batch_row is a demo batch position; use a persistent customer/event key in production.


## Practice

Turn ingestion, training and scoring into three Lakeflow Job tasks. Pass an immutable model URI and feature table version between tasks.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Use task values or a run-manifest Delta table. Configure retries for idempotent ingestion/scoring, timeouts, validation gates and failure notifications. Never share Python variables implicitly between job tasks.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
